# Week 15, Day 1: Random variables and distributions: describing variable responses

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w015/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A PMF from data

Here are spike counts from 2,000 presentations of one stimulus. Build the empirical PMF `pmf` (index k holds the fraction of trials with k spikes) and CDF `cdf`. Use them to find `p_four` = P(X = 4), `p_five_or_more` = P(X ≥ 5) via the complement rule, and the `median`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1983)
counts = rng.poisson(4.0, 2000)

pmf = ...                       # np.bincount helps
cdf = ...
p_four = ...
p_five_or_more = ...            # 1 minus P(X <= 4)
median = ...
print(f"P(X = 4) = {p_four:.3f},  P(X != 4) = {1 - p_four:.3f},  P(X >= 5) = {p_five_or_more:.3f},  median = {median}")
plt.bar(np.arange(len(pmf)), pmf, label="PMF"); plt.step(np.arange(len(cdf)), cdf, where="post", color="k", label="CDF")
plt.xlabel("spike count"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_pmf():
    "pmf sums to 1 and matches the relative frequencies"
    assert abs(pmf.sum() - 1) < 1e-12 and abs(pmf[4] - np.mean(counts == 4)) < 1e-12

def test_cdf():
    "cdf is the running sum of the pmf"
    assert np.allclose(cdf, np.cumsum(pmf))

def test_events():
    "P(X = 4) and P(X >= 5) are computed correctly from the pmf and cdf"
    assert abs(p_four - np.mean(counts == 4)) < 1e-12
    assert abs(p_five_or_more - np.mean(counts >= 5)) < 1e-12

def test_median():
    "the median is the smallest k with cdf at least 0.5"
    assert median == int(np.argmax(cdf >= 0.5))

test_pmf()
test_cdf()
test_events()
test_median()
print("All checks passed")

### Exercise: Shared gain makes neurons dependent

Two neurons share a random gain g on each trial. Build `joint(a, b)` from their counts, its marginals `pa` and `pb`, and the largest absolute difference between the joint table and the outer product of the marginals, `gap_shared`. Repeat with a fixed gain to get `gap_indep`.

In [ ]:
T = 20000
g = rng.gamma(5.0, 1 / 5.0, T)                  # shared gain: mean 1, varies from trial to trial
a = rng.poisson(3 * g); b = rng.poisson(5 * g)

def joint_table(a, b):
    J = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(J, (a, b), 1)
    return J / len(a)

joint = joint_table(a, b)
pa, pb = ..., ...
gap_shared = ...

a2, b2 = rng.poisson(3, T), rng.poisson(5, T)    # fixed gain
J2 = joint_table(a2, b2)
gap_indep = ...
print(f"largest |joint - product|: shared gain {gap_shared:.4f}, fixed gain {gap_indep:.4f}")
print("P(both silent) =", round(joint[0, 0], 4), " product of marginals =", round(pa[0] * pb[0], 4))

In [ ]:
import numpy as np
def test_joint():
    "joint is a normalised 2-D table of relative frequencies"
    assert abs(joint.sum() - 1) < 1e-12 and abs(joint[0, 0] - np.mean((a == 0) & (b == 0))) < 1e-12

def test_marginals():
    "the marginals are the row and column sums"
    assert np.allclose(pa, joint.sum(1)) and np.allclose(pb, joint.sum(0))

def test_dependence():
    "with a shared gain, the joint differs from the product of marginals much more than for independent neurons"
    assert gap_shared > 3 * gap_indep

test_joint()
test_marginals()
test_dependence()
print("All checks passed")

### Exercise: One over root n

For each n in `ns`, draw 2,000 independent samples of size n from a Poisson distribution with mean 4, compute each sample's mean, and store the standard deviation of those means in `errors`. Fit the slope of log(error) against log(n).

In [ ]:
ns = np.array([4, 10, 25, 100, 400, 1600])
errors = []
for n in ns:
    means = ...                                    # 2000 sample means, each from n draws
    errors.append(...)
errors = np.array(errors)
slope = np.polyfit(np.log(ns), np.log(errors), 1)[0]
print("errors:", errors.round(4), " slope:", round(slope, 3), " sigma / sqrt(n):", (2 / np.sqrt(ns)).round(4))
plt.loglog(ns, errors, "o-", label="measured"); plt.loglog(ns, 2 / np.sqrt(ns), "k--", label="σ/√n"); plt.xlabel("n"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_errors():
    "errors has one entry per n, each the standard deviation of 2000 sample means"
    assert len(errors) == len(ns) and np.all(np.diff(errors) < 0)

def test_slope():
    "on log-log axes the error falls with slope close to -1/2"
    assert abs(slope + 0.5) < 0.05

def test_formula():
    "the error at n = 100 is close to sigma / sqrt(100) = 0.2"
    assert abs(errors[list(ns).index(100)] - 0.2) < 0.02

test_errors()
test_slope()
test_formula()
print("All checks passed")